# JSD Heatmap (Figure 2)

Reproduces **Figure 2** from the DTR paper.

This visualization shows a layer-by-token heatmap of Jensen-Shannon Divergence (JSD) values
between intermediate layer distributions and the final layer distribution. The heatmap reveals
how the model's internal representations "settle" across layers during generation, which is the
core intuition behind the Deep-Thinking Ratio (DTR) metric.

- **X-axis**: Token position in the generated sequence
- **Y-axis**: Layer index (bottom = early layers, top = later layers)
- **Color**: JSD value (high = layer distribution differs from final layer)

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

sns.set_theme(style="whitegrid")

from dtr.metrics.dtr import compute_dtr, DTRAccumulator
from dtr.metrics.distances import jsd
from dtr.utils.io import load_json, load_jsonl

## Configuration

Select which model and benchmark to visualize, and which sample index to display.

In [ ]:
MODEL = "deepseek-r1"  # or "qwq-32b"
BENCHMARK = "aime_2025"
SAMPLE_IDX = 0  # Index of the sample to visualize

RESULTS_DIR = Path("..") / "results" / "raw" / MODEL / BENCHMARK
print(f"Loading from: {RESULTS_DIR}")
print(f"Available files: {list(RESULTS_DIR.glob('*')) if RESULTS_DIR.exists() else 'directory not found'}")

## Load JSD Matrix

The JSD matrix has shape `(num_layers, num_tokens)` and contains the JSD between each
layer's vocabulary distribution and the final layer's distribution at each token position.

In [ ]:
# Load the JSD matrix for the selected sample
jsd_files = sorted(RESULTS_DIR.glob("jsd_matrix_*.pt"))
if not jsd_files:
    # Try alternative naming
    jsd_files = sorted(RESULTS_DIR.glob("*.pt"))

print(f"Found {len(jsd_files)} JSD matrix files")

jsd_matrix = torch.load(jsd_files[SAMPLE_IDX], map_location="cpu")
if isinstance(jsd_matrix, dict):
    # May be stored as a dict with metadata
    jsd_matrix = jsd_matrix.get("jsd_matrix", jsd_matrix.get("matrix"))

jsd_matrix = jsd_matrix.numpy() if isinstance(jsd_matrix, torch.Tensor) else np.array(jsd_matrix)
print(f"JSD matrix shape: {jsd_matrix.shape}  (layers x tokens)")
print(f"Value range: [{jsd_matrix.min():.4f}, {jsd_matrix.max():.4f}]")

## Load Token Information

Load the generated tokens to annotate functional vs. answer tokens on the heatmap.

In [ ]:
# Load generation metadata with tokens
meta_files = sorted(RESULTS_DIR.glob("generation_*.json")) + sorted(RESULTS_DIR.glob("*.jsonl"))
tokens = None

if meta_files:
    meta = load_json(str(meta_files[0])) if meta_files[0].suffix == ".json" else load_jsonl(str(meta_files[0]))
    if isinstance(meta, list):
        meta = meta[SAMPLE_IDX] if SAMPLE_IDX < len(meta) else meta[0]
    tokens = meta.get("tokens", meta.get("generated_tokens", None))

if tokens is not None:
    print(f"Loaded {len(tokens)} tokens")
    print(f"First 20 tokens: {tokens[:20]}")
else:
    print("Token information not available; heatmap will use position indices only.")

## Main Heatmap (Figure 2)

The core visualization: a layer x token heatmap of JSD values.

- High JSD (warm colors) indicates the layer's prediction differs significantly from the final layer
- Low JSD (cool colors) indicates the layer has already "settled" to the final prediction
- The settling depth (white dashed line) shows where each token's JSD first drops below a threshold

In [ ]:
num_layers, num_tokens = jsd_matrix.shape

# For readability, limit the number of displayed tokens if sequence is very long
MAX_DISPLAY_TOKENS = 200
display_matrix = jsd_matrix[:, :MAX_DISPLAY_TOKENS]
display_tokens = num_tokens if num_tokens <= MAX_DISPLAY_TOKENS else MAX_DISPLAY_TOKENS

fig, ax = plt.subplots(figsize=(14, 8))

# Create heatmap with diverging colormap
im = ax.imshow(
    display_matrix,
    aspect="auto",
    cmap="RdBu_r",
    interpolation="nearest",
    origin="lower",
    vmin=0,
    vmax=np.percentile(display_matrix, 99),
)

# Add colorbar
cbar = fig.colorbar(im, ax=ax, label="JSD", shrink=0.8, pad=0.02)
cbar.ax.tick_params(labelsize=10)

# Compute and overlay settling depth line
# Settling depth: for each token, find the deepest layer where JSD > threshold
THRESHOLD = 0.1
settling_depth = np.zeros(display_tokens)
for t in range(display_tokens):
    above_threshold = np.where(display_matrix[:, t] > THRESHOLD)[0]
    settling_depth[t] = above_threshold[-1] if len(above_threshold) > 0 else 0

ax.plot(
    range(display_tokens),
    settling_depth,
    color="white",
    linewidth=2,
    linestyle="--",
    alpha=0.8,
    label=f"Settling depth (JSD > {THRESHOLD})",
)
ax.legend(loc="upper right", fontsize=10, framealpha=0.9)

# Annotate functional vs answer tokens if available
if tokens is not None:
    FUNCTIONAL_TOKENS = {"the", "a", "an", "of", "in", "to", "for", "is", "and", "or", "that", "with"}
    OPERATOR_TOKENS = {"+", "-", "*", "/", "=", ">", "<", "(", ")"}
    
    for i, tok in enumerate(tokens[:display_tokens]):
        tok_clean = tok.strip().lower()
        if tok_clean in FUNCTIONAL_TOKENS:
            ax.axvline(x=i, color="cyan", alpha=0.15, linewidth=1)
        elif tok_clean in OPERATOR_TOKENS or tok_clean.replace(".", "").isdigit():
            ax.axvline(x=i, color="yellow", alpha=0.15, linewidth=1)

# Labels and formatting
ax.set_xlabel("Token Position", fontsize=13)
ax.set_ylabel("Layer Index", fontsize=13)
ax.set_title(
    f"JSD Heatmap: {MODEL} on {BENCHMARK} (Sample {SAMPLE_IDX})",
    fontsize=15,
    fontweight="bold",
)

# Reduce tick clutter
ax.xaxis.set_major_locator(ticker.MaxNLocator(20))
ax.yaxis.set_major_locator(ticker.MaxNLocator(15))

plt.tight_layout()
plt.savefig("../results/figures/fig2_jsd_heatmap.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Zoomed View: First 50 Tokens

A detailed view of the early token positions where the thinking structure is most visible.

In [ ]:
ZOOM_TOKENS = min(50, num_tokens)
zoom_matrix = jsd_matrix[:, :ZOOM_TOKENS]

fig, ax = plt.subplots(figsize=(14, 8))

sns_heatmap = sns.heatmap(
    zoom_matrix,
    ax=ax,
    cmap="coolwarm",
    xticklabels=tokens[:ZOOM_TOKENS] if tokens else True,
    yticklabels=5,
    cbar_kws={"label": "JSD", "shrink": 0.8},
)

ax.set_xlabel("Token", fontsize=13)
ax.set_ylabel("Layer Index", fontsize=13)
ax.set_title(
    f"JSD Heatmap (Zoomed): First {ZOOM_TOKENS} Tokens",
    fontsize=15,
    fontweight="bold",
)

if tokens:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)

plt.tight_layout()
plt.show()

## Layer-Averaged JSD Profile

Average JSD across all tokens for each layer. This shows the general settling pattern:
early layers have high JSD (different from final) while later layers converge.

In [ ]:
layer_avg_jsd = jsd_matrix.mean(axis=1)
layer_std_jsd = jsd_matrix.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))

layers = np.arange(num_layers)
ax.plot(layers, layer_avg_jsd, color=sns.color_palette()[0], linewidth=2, label="Mean JSD")
ax.fill_between(
    layers,
    layer_avg_jsd - layer_std_jsd,
    layer_avg_jsd + layer_std_jsd,
    alpha=0.2,
    color=sns.color_palette()[0],
    label="+/- 1 std",
)

ax.set_xlabel("Layer Index", fontsize=13)
ax.set_ylabel("JSD (vs. final layer)", fontsize=13)
ax.set_title("Layer-Averaged JSD Profile", fontsize=15, fontweight="bold")
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()